# Cross Validation Classification

In [ ]:
import sklearn as sk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Importing data

In [ ]:
df = pd.read_parquet("advanced_processed_data.parquet")

## Preprocessing

In [ ]:
data = df.copy().drop("Severity", axis = 1)
target = df.copy()["Severity"]

classes = sorted(target.unique())

### Helper Functions

In [ ]:
def print_results(fold, y_test,prediction, classes):
    confusion = sk.metrics.confusion_matrix(y_test,prediction)
    accuracy = sk.metrics.accuracy_score(y_test,prediction)
    precision, recall, f1, support = sk.metrics.precision_recall_fscore_support(y_test,prediction,zero_division=0)

    print(f"fold: {fold}")
    print("Confusion Matrix")
    print(confusion)
    print(f"Accuracy: {accuracy:.3f}")

    for i, severity in enumerate([0,1,2,3]):
        print(f"Severity: {severity}:",
            f"Precision: {precision[i]:.3f}",
            f"Recall: {recall[i]:.3f}",
            f"F1: {f1[i]:.3f}")

    return confusion, accuracy, precision, recall, f1, support

def collect_details(model, data, target, skf,classes):
    rows = []
    confusions = []

    for fold, (train_index, test_index) in enumerate(skf.split(data, target), start = 1):
        X_train, X_test = data.iloc[train_index], data.iloc[test_index]
        y_train, y_test = target.iloc[train_index], target.iloc[test_index]

        model.fit(X_train,y_train)
        prediction = model.predict(X_test)

        confusion, accuracy, precision, recall, f1, support = print_results(fold, y_test, prediction, classes)

        confusions.append(confusion)

        for i, severity in enumerate(target.unique()):
            rows.append({
                "Fold": fold,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "F1_score": f1,
                "Support": support
            })

    return pd.DataFrame(rows), confusions


## 10 Fold Stratified Cross Validation

In [ ]:
skf = sk.model_selection.StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

### Dummy Classifier

In [ ]:
dummy = sk.dummy.DummyClassifier(strategy="most_frequent")
dummy_cv = sk.model_selection.cross_validate(dummy,data,target,cv=skf,scoring=scoring,return_train_score=False)

print("Dummy Classifier")
print(f"Accuracy mean: {dummy_cv['test_accuracy'].mean():.3f}")
print(f"Accuracy std: {dummy_cv['test_accuracy'].std():.3f}")
print(f"Precision mean: {dummy_cv['test_precision_macro'].mean():.3f}")
print(f"Precision std: {dummy_cv['test_precision_macro'].std():.3f}")
print(f"Recall mean: {dummy_cv['test_recall_macro'].mean():.3f}")
print(f"Recall std: {dummy_cv['test_recall_macro'].std():.3f}")
print(f"F1 mean: {dummy_cv['test_f1_macro'].mean():.3f}")
print(f"F1 std: {dummy_cv['test_f1_macro'].std():.3f}")

In [ ]:
dummy_details, dummy_confusion_matricies = collect_details(dummy,data,target,skf,classes)
#dummy_details

## max_depth=None Decision Tree

In [ ]:
tree = sk.tree.DecisionTreeClassifier(max_depth=10)
tree_cv = sk.model_selection.cross_validate(tree,data,target,cv=skf,scoring=scoring,return_train_score=False)

print("max_depth=None Decision Tree Classifier")
print(f"Accuracy mean: {tree_cv['test_accuracy'].mean():.3f}")
print(f"Accuracy std: {tree_cv['test_accuracy'].std():.3f}")
print(f"Precision mean: {tree_cv['test_precision_macro'].mean():.3f}")
print(f"Precision std: {tree_cv['test_precision_macro'].std():.3f}")
print(f"Recall mean: {tree_cv['test_recall_macro'].mean():.3f}")
print(f"Recall std: {tree_cv['test_recall_macro'].std():.3f}")
print(f"F1 mean: {tree_cv['test_f1_macro'].mean():.3f}")
print(f"F1 std: {tree_cv['test_f1_macro'].std():.3f}")

In [ ]:
tree_details, tree_confusion_matricies = collect_details(tree,data,target,skf,classes)
#dummy_details


## max_depth=5 Decision Tree

In [ ]:
tree_5 = sk.tree.DecisionTreeClassifier(max_depth=5)
tree_5_cv = sk.model_selection.cross_validate(tree_5,data,target,cv=skf,scoring=scoring,return_train_score=False)

print("max_depth=None Decision Tree Classifier")
print(f"Accuracy mean: {tree_cv['test_accuracy'].mean():.3f}")
print(f"Accuracy std: {tree_cv['test_accuracy'].std():.3f}")
print(f"Precision mean: {tree_cv['test_precision_macro'].mean():.3f}")
print(f"Precision std: {tree_cv['test_precision_macro'].std():.3f}")
print(f"Recall mean: {tree_cv['test_recall_macro'].mean():.3f}")
print(f"Recall std: {tree_cv['test_recall_macro'].std():.3f}")
print(f"F1 mean: {tree_cv['test_f1_macro'].mean():.3f}")
print(f"F1 std: {tree_cv['test_f1_macro'].std():.3f}")

In [ ]:
tree_5_details, tree_5_confusion_matricies = collect_details(tree_5,data,target,skf,classes)
#dummy_details


## max_depth=None Random Forest Classifier

In [ ]:
forest = sk.ensemble.RandomForestClassifier(max_depth=10)
forest_cv = sk.model_selection.cross_validate(forest,data,target,cv=skf,scoring=scoring,return_train_score=False)

print("max_depth=None Decision Tree Classifier")
print(f"Accuracy mean: {tree_cv['test_accuracy'].mean():.3f}")
print(f"Accuracy std: {tree_cv['test_accuracy'].std():.3f}")
print(f"Precision mean: {tree_cv['test_precision_macro'].mean():.3f}")
print(f"Precision std: {tree_cv['test_precision_macro'].std():.3f}")
print(f"Recall mean: {tree_cv['test_recall_macro'].mean():.3f}")
print(f"Recall std: {tree_cv['test_recall_macro'].std():.3f}")
print(f"F1 mean: {tree_cv['test_f1_macro'].mean():.3f}")
print(f"F1 std: {tree_cv['test_f1_macro'].std():.3f}")

In [ ]:
forest_details, forest_confusion_matricies = collect_details(forest,data,target,skf,classes)
#dummy_details


## max_depth=5 Random Forest Classifier

In [ ]:
forest_5 = sk.ensemble.RandomForestClassifier(max_depth=5)
forest_5_cv = sk.model_selection.cross_validate(forest_5,data,target,cv=skf,scoring=scoring,return_train_score=False)

print("max_depth=None Decision Tree Classifier")
print(f"Accuracy mean: {tree_cv['test_accuracy'].mean():.3f}")
print(f"Accuracy std: {tree_cv['test_accuracy'].std():.3f}")
print(f"Precision mean: {tree_cv['test_precision_macro'].mean():.3f}")
print(f"Precision std: {tree_cv['test_precision_macro'].std():.3f}")
print(f"Recall mean: {tree_cv['test_recall_macro'].mean():.3f}")
print(f"Recall std: {tree_cv['test_recall_macro'].std():.3f}")
print(f"F1 mean: {tree_cv['test_f1_macro'].mean():.3f}")
print(f"F1 std: {tree_cv['test_f1_macro'].std():.3f}")

In [ ]:
forest_5_details, forest_5_confusion_matricies = collect_details(forest_5,data,target,skf,classes)

## Comparison

In [ ]:
cross_validation_comparison = {
    "Dummy": dummy_cv,
    "max_depth=None Decision Tree": tree_cv,
    "max_depth=5 Decision Tree": tree_5_cv,
    "max_depth=None Random Forest",forest_cv,
    "max_depth=5 Random Forest", forest_5_cv
}